# 3. Predict — original CRM data (`data/crm/`)

Same steps as `python -m src.predict --output reports/crm/final`, using the functions in `src/predict.py`.
Uses the frozen calibrated model saved by `1_train.ipynb`. Open deals are never labelled Lost.

In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "train.py").is_file())
sys.path.insert(0, str(ROOT))

import tempfile
import pandas as pd
from src import predict, train
from src.outputs import Outputs
from src.data.crm import CRM_DIR


## Settings

Must match the settings used in `1_train.ipynb`.

In [2]:
DATA_DIR = CRM_DIR   # real Maven CRM tables
QUICK = False            # True: tiny budgets, smoke test only (writes a *smoke* folder)
OUTPUT = train.default_output(DATA_DIR, QUICK)
print("Data:  ", DATA_DIR)
print("Output:", OUTPUT)

Data:   /Users/thaiphan/sources/miu/cs582-project/data/crm
Output: /Users/thaiphan/sources/miu/cs582-project/reports/crm/final


## 1. Load the scoring bundle

In [3]:
out = Outputs(OUTPUT)
scoring = predict.load(out)
print("Model:", scoring["model"].name)
print("Decision threshold (calibrated):", round(scoring["decision_threshold"], 4))
print("Features:", len(scoring["features"]), "| data:", out.source_data())

Model: logistic_regression_calibrated
Decision threshold (calibrated): 0.6044
Features: 18 | data: /Users/thaiphan/sources/miu/cs582-project/data/crm


## 2. Score the open Engaging deals

Writes `predictions/open_deal_predictions.csv`: P(Won), P(Lost), predicted class, a heuristic priority band,
the inputs that raise or lower each score, and a warning when the account is missing.

In [4]:
scores = predict.run(OUTPUT)
pd.Series({
    "scored deals": len(scores),
    "predicted Won": int(scores.predicted_outcome.eq("Won").sum()),
    "High / Medium / Low": "/".join(str(int(scores.priority.eq(p).sum())) for p in ["High", "Medium", "Low"]),
    "missing account (warning)": int(scores.account_missing.sum()),
    "mean P(Won)": round(scores.win_probability.mean(), 4),
})

PREDICTED: 1,589 open deals -> /Users/thaiphan/sources/miu/cs582-project/reports/crm/final/predictions/open_deal_predictions.csv


scored deals                       1589
predicted Won                      1025
High / Medium / Low          400/1189/0
missing account (warning)          1088
mean P(Won)                      0.6397
dtype: object

In [5]:
scores[["opportunity_id", "product", "sales_agent", "win_probability", "priority",
        "positive_factors", "negative_factors", "input_warning"]].head(10)

,opportunity_id,product,sales_agent,win_probability,priority,positive_factors,negative_factors,input_warning
0,AI76U58A,GTX Basic,Maureen Marcano,0.879359,High,engage_year=2016.0 (delta P(Won)=+0.2936); sal...,engage_quarter=4.0 (delta P(Won)=-0.0635); sec...,
1,0DRC1U9Q,GTX Basic,Maureen Marcano,0.875158,High,engage_year=2016.0 (delta P(Won)=+0.2989); sal...,engage_quarter=4.0 (delta P(Won)=-0.0655); yea...,
2,ZZY4516R,MG Advanced,Hayden Neloms,0.874432,High,engage_year=2016.0 (delta P(Won)=+0.2998); sal...,engage_quarter=4.0 (delta P(Won)=-0.0659); off...,Account missing: numeric imputation / unknown ...
3,579LZ3F9,GTX Plus Basic,Daniell Hammack,0.873588,High,engage_year=2016.0 (delta P(Won)=+0.3008); sal...,engage_quarter=4.0 (delta P(Won)=-0.0663); man...,
4,M7I5O9YU,GTX Basic,Corliss Cosme,0.872845,High,engage_year=2016.0 (delta P(Won)=+0.3017); eng...,engage_quarter=4.0 (delta P(Won)=-0.0666); eng...,
5,RJYG004O,GTX Basic,Daniell Hammack,0.872640,High,engage_year=2016.0 (delta P(Won)=+0.3019); eng...,engage_quarter=4.0 (delta P(Won)=-0.0667); man...,
6,ERFRHR3Z,MG Advanced,Hayden Neloms,0.870075,High,engage_year=2016.0 (delta P(Won)=+0.3050); sal...,engage_quarter=4.0 (delta P(Won)=-0.0680); off...,Account missing: numeric imputation / unknown ...
7,N07K7DR7,MG Special,Rosie Papadopoulos,0.865937,High,engage_year=2016.0 (delta P(Won)=+0.3097); eng...,engage_quarter=4.0 (delta P(Won)=-0.0700); ser...,
8,MNVMY83P,GTX Pro,Cassey Cress,0.865747,High,engage_year=2016.0 (delta P(Won)=+0.3100); eng...,engage_quarter=4.0 (delta P(Won)=-0.0701); yea...,
9,ICMXZOVL,GTX Pro,Daniell Hammack,0.863781,High,engage_year=2016.0 (delta P(Won)=+0.3122); eng...,engage_quarter=4.0 (delta P(Won)=-0.0710); sec...,


## 3. Score new deals from a CSV

New deals use the `sales_pipeline.csv` format: `opportunity_id, sales_agent, product, account, engage_date`.
This example takes five Engaging deals from the data as stand-ins; replace `new_deals` with your own file.

In [6]:
raw = pd.read_csv(out.source_data() / "sales_pipeline.csv")
new_deals = raw.loc[raw.deal_stage.eq("Engaging") & raw.account.notna(),
                    ["opportunity_id", "sales_agent", "product", "account", "engage_date"]].head(5)
with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "new_deals.csv"
    new_deals.to_csv(path, index=False)
    new_scores = predict.score_file(path, OUTPUT, Path(folder) / "new_deal_scores.csv")
new_scores[["opportunity_id", "win_probability", "predicted_outcome", "priority", "positive_factors"]]

PREDICTED: 5 deals -> /var/folders/xb/q75wg0yn2gl2bysx0cgshv4w0000gn/T/tmpyn6nhh2_/new_deal_scores.csv


,opportunity_id,win_probability,predicted_outcome,priority,positive_factors
0,0DRC1U9Q,0.875158,Won,High,engage_year=2016 (delta P(Won)=+0.2989); sales...
1,579LZ3F9,0.873588,Won,High,engage_year=2016 (delta P(Won)=+0.3008); sales...
2,M7I5O9YU,0.872845,Won,High,engage_year=2016 (delta P(Won)=+0.3017); engag...
3,VDIU10RV,0.819819,Won,High,engage_year=2016 (delta P(Won)=+0.3509); engag...
4,UP409DSB,0.802362,Won,High,engage_year=2016 (delta P(Won)=+0.3617); sales...
